<div style='background:#8B0000; padding:20px; border-radius:8px; margin-bottom:10px'>
<h1 style='color:white; text-align:center; margin:0'>Reconocimiento de Patrones</h1>
<h2 style='color:#ffcccc; text-align:center; margin:5px 0 0 0'>Challenge 4 — Árboles de Decisión, Random Forest y XGBoost</h2>
<p style='color:#ffaaaa; text-align:center; margin:5px 0 0 0'>Ingeniería Biomédica · UPCH · 2026-1</p>
</div>

## Contexto clínico

Retomamos el dataset **Heart Failure Prediction** (Kaggle, fedesoriano, 2021) que ya conoces del Challenge 1.
En ese challenge lo exploraste y preprocesaste; aquí tu misión es **construir, analizar y comparar modelos basados en árboles**.

La pregunta clínica sigue siendo la misma: dado el perfil de un paciente (**Edad, Colesterol, ST_Slope, MaxHR...**), ¿podemos predecir si tiene enfermedad cardíaca?

Los árboles de decisión son especialmente poderosos en datos biomédicos tabulares porque:
- **Son interpretables**: el médico puede trazar exactamente cómo se llegó a un diagnóstico.
- **Capturan no-linealidades** y umbrales clínicos (ej: Oldpeak > 1.5 → riesgo elevado).
- **Random Forest y XGBoost** extienden esta idea con ensambles que reducen el sobreajuste.

---
> **Dataset:** `heart.csv` (ya descargado desde el Challenge 1)  
> **Fuente:** fedesoriano. (2021). *Heart Failure Prediction Dataset*. Kaggle. https://www.kaggle.com/fedesoriano/heart-failure-prediction  
> **Variables:** Age, Sex, ChestPainType, RestingBP, Cholesterol, FastingBS, RestingECG, MaxHR, ExerciseAngina, Oldpeak, ST_Slope → **HeartDisease** (0/1)

### Estructura de carpetas
```
Clase5_Challenge4/
├── Challenge4_ApellidoNombre.ipynb
└── heart.csv     ← el mismo del Challenge 1
```

---
---
## Ejercicio 1 — Diseño previo al código

### 1.1 — Diagnóstico esperado

<div style='background:#e8f5e9; padding:8px 14px; border-left:4px solid #2e7d32; border-radius:4px; margin:8px 0'>
⏱️ <b>EN CLASE</b> — completa esto durante la sesión
</div>

Antes de escribir una línea de código, responde las siguientes preguntas:

**1. ¿Un árbol de decisión profundo tiene más sesgo o más varianza que un árbol *prepodado* (profundidad controlada)? Relaciona con la descomposición sesgo-varianza:**

$$\text{Error esperado}(x) = \underbrace{\left(y^\star - \mathbb{E}[y]\right)^2}_{\text{sesgo}^2} + \underbrace{\text{Var}(y)}_{\text{varianza}} + \underbrace{\text{Var}(t)}_{\text{error de Bayes}}$$

**📝 Tu respuesta (edita esta celda):**  
Árbol profundo   → sesgo ___ / varianza ___  
Árbol prepodado → sesgo ___ / varianza ___  
Justificación: ___

---

**2. ¿Qué diferencia hay entre Gini e Entropía como criterios de división? ¿Cuándo llevan a árboles distintos?**

| Criterio | Fórmula | Rango (2 clases) |
|---|---|---|
| Gini | ___ | ___ |
| Entropía | ___ | ___ |

**📝 Tu respuesta:** ___

---

**3. ¿Por qué Random Forest reduce la varianza pero NO el sesgo?**

Pista: $\bar{y} = \frac{1}{M}\sum_{m=1}^M y_m$ — ¿qué pasa con $\mathbb{E}[\bar{y}]$ y con $\text{Var}(\bar{y})$?

**📝 Tu respuesta:** ___

---

**4. Completa la tabla comparando bagging (RF) y boosting (XGBoost):**

| | Bagging / RF | Boosting / XGBoost |
|---|---|---|
| Entrenamiento | ___ | ___ |
| Objetivo (sesgo/varianza) | ___ | ___ |
| Riesgo principal | ___ | ___ |

---

**5. ¿Qué features del dataset heart.csv (que ya conoces del Challenge 1) esperas que sean las más importantes para predecir HeartDisease? Justifica clínicamente.**

**📝 Tu respuesta:** ___


### 1.2 — Matemática de la impureza

<div style='background:#e8f5e9; padding:8px 14px; border-left:4px solid #2e7d32; border-radius:4px; margin:8px 0'>
⏱️ <b>EN CLASE</b> — completa esto durante la sesión
</div>

Usando las fórmulas de **Gini** e **Entropía**:

$$G(n) = 1 - \sum_k p_k^2 \qquad H(n) = -\sum_k p_k \log_2 p_k$$

**Calcula para un nodo con 20 pacientes: 10 con enf. (pos), 10 sin enf. (neg):**

$$G(\text{raíz}) = 1 - (\_\_)^2 - (\_\_)^2 = \_\_$$

$$H(\text{raíz}) = -(\_\_)\log_2(\_\_) - (\_\_)\log_2(\_\_) = \_\_ \text{ bits}$$

---

**Después de dividir por "ST\_Slope\_Up = 1":**

| Rama | Muestras | $p_+$ | Gini | Entropía |
|---|---|---|---|---|
| Izq (ST_Slope_Up=1) | 10 | ___ | ___ | ___ |
| Der (ST_Slope_Up=0) | 10 | ___ | ___ | ___ |

**Reducción de impureza / ganancia de información:**

$$\Delta Gini = G(\text{raíz}) - \left(\frac{10}{20}\times G(\text{Izq}) + \frac{10}{20}\times G(\text{Der})\right) = \_\_$$

$$IG_H = H(\text{raíz}) - \left(\frac{10}{20}\times H(\text{Izq}) + \frac{10}{20}\times H(\text{Der})\right) = \_\_ \text{ bits}$$

**📝 Completa los blancos arriba.**


### 1.3 — Pipeline completo

Completa el diagrama de pipeline para este challenge:

```
heart.csv
    │
    ▼
[ Paso A ] _______________________________________________
           _______________________________________________
    │
    ▼
[ Paso B ] Train/Test split estratificado (80/20)
           ← AQUÍ se divide. ¿Por qué es importante este orden?
           📝 Tu respuesta: ___
    │
    ├─── TRAIN (80%) ──────────────────────────────────────┐
    │                                                     │
    ▼                                                     │
[ Paso C ] ___                                           │
    │                                                     │
    ▼                                                     │
[ Paso D ] Modelos: _____________________________________ │
    │                                                     │
    └─── TEST (20%) ── ___ ──────────────────────────── ──┘
    │
    ▼
[ Paso G ] _______________________________________________
```

**📝 Tu respuesta (edita esta celda):** completa los blancos del diagrama.

---
## Ejercicio 2 — Setup y carga de datos

<div style='background:#e8f5e9; padding:8px 14px; border-left:4px solid #2e7d32; border-radius:4px; margin:8px 0'>
⏱️ <b>EN CLASE</b> — completa esto durante la sesión
</div>

In [ ]:
# ── Conexión con Google Drive ──────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

### Paso 2.1 — Librerías

In [ ]:
UPCH_RED    = '#8B0000'
UPCH_BLUE   = '#1565C0'
UPCH_GREEN  = '#2E7D32'
UPCH_ORANGE = '#E65100'

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# TODO 1: importa DecisionTreeClassifier, plot_tree desde sklearn.tree
from sklearn.tree import ___

# TODO 2: importa RandomForestClassifier desde sklearn.ensemble
from sklearn.ensemble import ___

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (train_test_split, cross_val_score,
                                      StratifiedKFold)
from sklearn.metrics import (accuracy_score, recall_score, precision_score,
                              f1_score, confusion_matrix, ConfusionMatrixDisplay,
                              roc_curve, auc, classification_report)

# XGBoost
try:
    from xgboost import XGBClassifier
    print('XGBoost disponible ✓')
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'xgboost', '-q'])
    from xgboost import XGBClassifier
    print('XGBoost instalado ✓')

plt.rcParams['figure.dpi'] = 120
BASE = Path('COLOCA TU RUTA AQUÍ')
print('Setup completo ✓')

### Paso 2.2 — Carga del dataset

In [ ]:
# ── Carga del dataset ──────────────────────────────────────────
df_raw = pd.read_csv(BASE / 'heart.csv')
print(f'Shape: {df_raw.shape}')
print(f'Columnas: {list(df_raw.columns)}')
df_raw.head()

### Paso 2.3 — Preprocesamiento compacto

In [ ]:
# ── Preprocesamiento compacto (recapitulando Challenge 1) ──────
df = df_raw.copy()

# 1. Cholesterol = 0 → NaN (imposible fisiológicamente)
df.loc[df['Cholesterol'] == 0, 'Cholesterol'] = np.nan

# 2. One-Hot Encoding
cat_cols = ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# 3. Separar features y target
X_raw         = df.drop(columns='HeartDisease').values.astype(np.float64)
y             = df['HeartDisease'].values.astype(int)
feature_names = df.drop(columns='HeartDisease').columns.tolist()

# 4. Train/Test split estratificado (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

# TODO 3: imputar Cholesterol con la mediana de TRAIN (usa SimpleImputer)
# ⚠️ IMPORTANTE: fit SOLO sobre X_train, luego transform sobre X_test
imputer = SimpleImputer(strategy='median')
X_train = ___           # fit_transform sobre train
X_test  = ___           # transform (sin fit) sobre test

# TODO 4: escalar con StandardScaler — fit SOLO en train
scaler     = StandardScaler()
X_train_sc = ___
X_test_sc  = ___

print(f'Train: {X_train.shape}  (0: {(y_train==0).sum()}, 1: {(y_train==1).sum()})')
print(f'Test:  {X_test.shape}   (0: {(y_test==0).sum()},  1: {(y_test==1).sum()})')
print(f'Features ({len(feature_names)}): {feature_names[:5]} ...')

---
# PARTE 1 — Árbol de Decisión
## Ejercicio 3 — Árbol base y curva de profundidad

<div style='background:#e8f5e9; padding:8px 14px; border-left:4px solid #2e7d32; border-radius:4px; margin:8px 0'>
⏱️ <b>EN CLASE</b> — completa esto durante la sesión
</div>

### Paso 3.1 — Árbol base (max_depth=3) y visualización

In [ ]:
# ── Árbol de decisión base (max_depth=3) ───────────────────────
# ⚠️ Los árboles comparan UMBRALES, no distancias → no requieren escalado
# Usamos X_train / X_test (sin escalar)

# TODO 5: crea un DecisionTreeClassifier con max_depth=3, criterion='gini',
#          min_samples_leaf=5, random_state=42
dt_base = ___

# TODO 6: entrena el árbol sobre X_train, y_train
___

y_pred_base = dt_base.predict(X_test)
acc_base    = accuracy_score(y_test, y_pred_base)
print(f'Accuracy DT (max_depth=3): {acc_base:.4f}')
print(f'Nodos totales: {dt_base.tree_.node_count}  |  Hojas: {dt_base.get_n_leaves()}')

# TODO 7: visualiza el árbol con plot_tree
#   usa: feature_names=feature_names, class_names=['Sin enf.','Con enf.'],
#        filled=True, rounded=True, fontsize=8
fig, ax = plt.subplots(figsize=(22, 8))
___
ax.set_title('Árbol de Decisión (max_depth=3) — Heart Disease',
             fontsize=14, fontweight='bold', color=UPCH_RED)
plt.tight_layout()
plt.savefig('dt_visualizacion.png', dpi=150, bbox_inches='tight')
plt.show()
print('→ ¿Qué feature aparece en la raíz del árbol? ¿Tiene sentido clínicamente?')

### Paso 3.2 — Curva de profundidad

In [ ]:
# ── Curva de profundidad: train vs test accuracy ────────────────
depths    = list(range(1, 21))
acc_train = []
acc_test  = []
acc_cv    = []

cv_depth = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for d in depths:
    # TODO 8: crea un DecisionTreeClassifier con max_depth=d,
    #         criterion='gini', min_samples_leaf=5, random_state=42
    dt = ___
    dt.fit(X_train, y_train)

    # TODO 9: agrega la accuracy de train y de test a las listas
    acc_train.append(___)
    acc_test.append(___)

    # TODO 10: calcula la CV accuracy media SOLO sobre TRAIN
    #   usa cross_val_score con cv=cv_depth y scoring='accuracy'
    sc_cv = ___
    acc_cv.append(sc_cv)

# Selección correcta del modelo: NO usar test para escoger max_depth
best_idx   = ___
best_depth = ___
print(f'Mejor max_depth (CV-5 en train): depth={best_depth}  →  acc_cv={acc_cv[best_idx]:.4f}')

# Gráfica
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(depths, acc_train, 'o--', color=UPCH_BLUE,  lw=2, label='Train')
ax.plot(depths, acc_test,  's-',  color=UPCH_RED,   lw=2, label='Test')
ax.plot(depths, acc_cv,    '^:',  color=UPCH_GREEN, lw=2, label='CV-5 (train)')
ax.axvline(best_depth, color=UPCH_GREEN, ls=':', lw=2.5,
           label=f'depth* = {best_depth}')
ax.set_xlabel('max_depth', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Efecto de max_depth — Árbol de Decisión · Heart Disease',
             fontsize=13, fontweight='bold', color=UPCH_RED)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(0.6, 1.05)
plt.tight_layout()
plt.savefig('dt_curva_profundidad.png', dpi=150, bbox_inches='tight')
plt.show()


### Paso 3.2 — Interpretación

**📝 Tu respuesta (edita esta celda):**

**1. ¿Por qué la accuracy de train sube mientras la de test cae al aumentar max_depth?**  
___

**2. ¿En qué profundidad empieza el sobreajuste? ¿Cómo lo identificas en la gráfica?**  
___

**3. ¿Qué analogía clínica tiene el sobreajuste de un árbol muy profundo?**  
___

### Paso 3.3 — Feature Importance

<div style='background:#fff3e0; padding:8px 14px; border-left:4px solid #e65100; border-radius:4px; margin:8px 0'>
📚 <b>TAREA</b> — completa esto antes de la siguiente clase
</div>

In [ ]:
# ── Árbol óptimo + Feature Importance ─────────────────────────
# TODO 11: crea y entrena el árbol con best_depth
dt_opt = DecisionTreeClassifier(max_depth=best_depth, criterion='gini',
                                 min_samples_leaf=5, random_state=42)
___

importances_dt = dt_opt.feature_importances_

# TODO 12: ordena las importancias de mayor a menor (usa np.argsort)
idx_sorted = ___

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(
    [feature_names[i] for i in idx_sorted[::-1]],
    importances_dt[idx_sorted[::-1]],
    color=[UPCH_RED if i == 0 else UPCH_BLUE
           for i in range(len(feature_names))][::-1],
    alpha=0.85
)
ax.set_xlabel('Importancia (reducción media de Gini)', fontsize=11)
ax.set_title(f'Feature Importance — Árbol de Decisión regularizado (depth={best_depth})',
             fontsize=12, fontweight='bold', color=UPCH_RED)
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('dt_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 5 features más importantes:')
for r, i in enumerate(idx_sorted[:5]):
    print(f'  {r+1}. {feature_names[i]:<40} {importances_dt[i]:.4f}')
print()
print('→ ¿Coincide con tus predicciones clínicas del Ejercicio 1.1?')

### Paso 3.4 — Métricas y matriz de confusión

In [ ]:
# ── Métricas finales Árbol de Decisión ──────────────────────────
y_pred_dt = dt_opt.predict(X_test)

# TODO 13: calcula las 4 métricas para el árbol óptimo
acc_dt  = ___
rec_dt  = ___
prec_dt = ___
f1_dt   = ___

print('=' * 52)
print(f'   Árbol de Decisión regularizado (max_depth={best_depth})')
print('=' * 52)
print(f'  Accuracy : {acc_dt:.4f}')
print(f'  Recall   : {rec_dt:.4f}')
print(f'  Precision: {prec_dt:.4f}')
print(f'  F1-Score : {f1_dt:.4f}')
print()
print(classification_report(y_test, y_pred_dt,
                             target_names=['Sin enf.', 'Con enf.']))

# TODO 14: grafica la matriz de confusión con ConfusionMatrixDisplay
fig, ax = plt.subplots(figsize=(5.5, 4.5))
___
ax.set_title(f'Árbol de Decisión regularizado (depth={best_depth})\nAccuracy = {acc_dt:.4f}',
             fontsize=11, fontweight='bold', color=UPCH_RED)
plt.tight_layout()
plt.savefig('cm_dt.png', dpi=150, bbox_inches='tight')
plt.show()

---
# PARTE 2 — Random Forest
## Ejercicio 4 — Efecto de n_estimators y optimización

<div style='background:#fff3e0; padding:8px 14px; border-left:4px solid #e65100; border-radius:4px; margin:8px 0'>
📚 <b>TAREA</b> — completa esto antes de la siguiente clase
</div>

### Paso 4.1 — Efecto de n_estimators

In [ ]:
# ── Efecto de n_estimators en Random Forest ────────────────────
n_list = [1, 5, 10, 20, 50, 100, 200, 300]
acc_rf_train = []
acc_rf_test  = []
acc_rf_oob   = []

for n in n_list:
    # TODO 15: crea un RandomForestClassifier con n_estimators=n,
    #           max_features='sqrt', oob_score=(n>=2), random_state=42, n_jobs=-1
    rf = ___
    rf.fit(X_train, y_train)

    acc_rf_train.append(accuracy_score(y_train, rf.predict(X_train)))
    acc_rf_test.append( accuracy_score(y_test,  rf.predict(X_test)))
    # TODO 16: agrega el OOB score (usa np.nan si n < 2)
    acc_rf_oob.append(___)
    oob_str = f'{acc_rf_oob[-1]:.4f}' if n >= 2 else 'N/A '
    print(f'n={n:>4}  train={acc_rf_train[-1]:.4f}  '
          f'test={acc_rf_test[-1]:.4f}  OOB={oob_str}')

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(n_list, acc_rf_train, 'o--', color=UPCH_BLUE,  lw=2, label='Train')
ax.plot(n_list, acc_rf_test,  's-',  color=UPCH_RED,   lw=2, label='Test')
ax.plot(n_list, acc_rf_oob,   '^:',  color=UPCH_GREEN, lw=2, label='OOB')
ax.set_xlabel('n_estimators', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Random Forest — Efecto de n_estimators · Heart Disease',
             fontsize=13, fontweight='bold', color=UPCH_RED)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('rf_n_estimators.png', dpi=150, bbox_inches='tight')
plt.show()

### Paso 4.1 — Interpretación

**📝 Tu respuesta (edita esta celda):**

**1. ¿Por qué la accuracy de RF converge al aumentar n_estimators?**  
Pista: $\text{Var}(\bar{y}) = \left(\frac{1-\rho}{M} + \rho\right)\sigma^2$  
___

**2. ¿Qué es el OOB score? ¿Por qué es útil?**  
___

**3. ¿RF tiene más o menos sesgo que un árbol individual del mismo max_depth?**  
___

### Paso 4.2 — Barrido max_features × max_depth

In [ ]:
# ── Random Forest optimizado (barrido max_features × max_depth) ─
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

mf_list = ['sqrt', 'log2', None]
md_list = [None, 5, 10, 15]

print('Barrido max_features × max_depth (CV-5):')
print(f'{"max_features":<15} {"max_depth":<12} {"CV acc"}')
print('-' * 42)

best_score_rf = 0
best_mf, best_md = 'sqrt', None

for mf in mf_list:
    for md in md_list:
        # TODO 17: crea RandomForestClassifier con los parámetros del loop
        rf_cv = ___
        # TODO 18: calcula la CV accuracy con cross_val_score
        sc = ___
        mf_s = str(mf) if mf else 'all'
        md_s = str(md) if md else 'None'
        print(f'{mf_s:<15} {md_s:<12} {sc:.4f}')
        if sc > best_score_rf:
            best_score_rf = sc
            best_mf, best_md = mf, md

print(f'\nMejores parámetros: max_features={best_mf}, max_depth={best_md}')
print(f'CV accuracy: {best_score_rf:.4f}')

### Paso 4.3 — Random Forest final

In [ ]:
# ── Random Forest final ─────────────────────────────────────────
# TODO 19: crea el RF final con n_estimators=200, best_mf, best_md,
#           oob_score=True, random_state=42, n_jobs=-1
rf_final = ___
rf_final.fit(X_train, y_train)

y_pred_rf = rf_final.predict(X_test)

# TODO 20: calcula las 4 métricas para RF
acc_rf  = ___
rec_rf  = ___
prec_rf = ___
f1_rf   = ___

print('=' * 52)
print(f'   Random Forest (n=200, mf={best_mf}, depth={best_md})')
print('=' * 52)
print(f'  OOB Score: {rf_final.oob_score_:.4f}')
print(f'  Accuracy : {acc_rf:.4f}')
print(f'  Recall   : {rec_rf:.4f}')
print(f'  Precision: {prec_rf:.4f}')
print(f'  F1-Score : {f1_rf:.4f}')
print(f'\n  Δ Accuracy vs DT: {acc_rf - acc_dt:+.4f}')

### Paso 4.4 — Feature Importance: DT vs RF

In [ ]:
# ── Feature Importance: DT vs RF ────────────────────────────────
# TODO 21: extrae las feature importances del RF final
importances_rf = ___

fig, axes = plt.subplots(1, 2, figsize=(17, 6))
for ax, imp, title, color in zip(
    axes,
    [importances_dt, importances_rf],
    [f'Árbol de Decisión regularizado (depth={best_depth})', 'Random Forest (200 árboles)'],
    [UPCH_BLUE, UPCH_RED]
):
    idx = np.argsort(imp)
    ax.barh([feature_names[i] for i in idx], imp[idx], color=color, alpha=0.8)
    ax.set_xlabel('Importancia', fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.grid(True, axis='x', alpha=0.3)

plt.suptitle('Feature Importance — DT vs Random Forest · Heart Disease',
             fontsize=13, fontweight='bold', color=UPCH_RED)
plt.tight_layout()
plt.savefig('fi_dt_vs_rf.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Correlación entre importancias DT y RF: '
      f'{np.corrcoef(importances_dt, importances_rf)[0,1]:.4f}')
print()
print('→ ¿Las RF importancias son más o menos estables que las del DT individual?')

---
# PARTE 3 — XGBoost
## Ejercicio 5 — Gradient Boosting

<div style='background:#fff3e0; padding:8px 14px; border-left:4px solid #e65100; border-radius:4px; margin:8px 0'>
📚 <b>TAREA</b> — completa esto antes de la siguiente clase
</div>

### Paso 5.1 — XGBoost base

In [ ]:
# ── XGBoost base ────────────────────────────────────────────────
# XGBoost actualiza secuencialmente: ĝ_m(x) = ĝ_{m-1}(x) + η·h_m(x)
# η = learning_rate; h_m = árbol que minimiza los residuos del paso anterior

# TODO 22: crea un XGBClassifier con n_estimators=100, learning_rate=0.1,
#           max_depth=4, subsample=0.8, colsample_bytree=0.8,
#           random_state=42, eval_metric='logloss', verbosity=0
xgb_base = ___

# TODO 23: entrena sobre X_train, y_train
___

y_pred_xgb_base = xgb_base.predict(X_test)
acc_xgb_base    = accuracy_score(y_test, y_pred_xgb_base)
print(f'Accuracy XGBoost base (lr=0.1, depth=4, n=100): {acc_xgb_base:.4f}')
print()
print('→ ¿XGBoost base supera al Árbol de Decisión y al Random Forest?')

### Paso 5.2 — Tuning: learning_rate × n_estimators

In [ ]:
# ── Tuning: barrido learning_rate × n_estimators ─────────────
lr_list    = [0.01, 0.05, 0.1, 0.2, 0.3]
n_est_list = [50, 100, 200, 300]

print('Barrido learning_rate × n_estimators (CV-5):')
print(f'{"lr":<8} {"n_est":<8} {"CV acc"}')
print('-' * 28)

best_score_xgb = 0
best_lr, best_n_xgb = 0.1, 100

for lr in lr_list:
    for n_est in n_est_list:
        # TODO 24: crea XGBClassifier con los parámetros del loop
        xgb_cv = ___

        # TODO 25: calcula CV accuracy (cv=cv, scoring='accuracy')
        sc = ___
        print(f'{lr:<8} {n_est:<8} {sc:.4f}')
        if sc > best_score_xgb:
            best_score_xgb = sc
            best_lr, best_n_xgb = lr, n_est

print(f'\nMejores parámetros: lr={best_lr}, n_estimators={best_n_xgb}')
print(f'CV accuracy: {best_score_xgb:.4f}')

### Paso 5.3 — XGBoost final

In [ ]:
# ── XGBoost final ───────────────────────────────────────────────
# TODO 26: crea el XGBClassifier final con los mejores parámetros
xgb_final = ___
xgb_final.fit(X_train, y_train)

y_pred_xgb = xgb_final.predict(X_test)

# TODO 27: calcula las 4 métricas para XGBoost
acc_xgb  = ___
rec_xgb  = ___
prec_xgb = ___
f1_xgb   = ___

print('=' * 55)
print(f'   XGBoost (lr={best_lr}, n={best_n_xgb}, depth=4)')
print('=' * 55)
print(f'  Accuracy : {acc_xgb:.4f}')
print(f'  Recall   : {rec_xgb:.4f}')
print(f'  Precision: {prec_xgb:.4f}')
print(f'  F1-Score : {f1_xgb:.4f}')
print(f'\n  Δ Accuracy vs DT: {acc_xgb - acc_dt:+.4f}')
print(f'  Δ Accuracy vs RF: {acc_xgb - acc_rf:+.4f}')

### Paso 5.4 — Feature Importance comparativa

In [ ]:
# ── Feature Importance: DT vs RF vs XGBoost ─────────────────────
# TODO 28: extrae las feature importances de xgb_final
importances_xgb = ___

fig, axes = plt.subplots(1, 3, figsize=(21, 6))
for ax, imp, title, color in zip(
    axes,
    [importances_dt, importances_rf, importances_xgb],
    [f'Árbol de Decisión regularizado\n(depth={best_depth})',
     'Random Forest\n(200 árboles)',
     f'XGBoost\n(lr={best_lr}, n={best_n_xgb})'],
    [UPCH_BLUE, UPCH_RED, UPCH_GREEN]
):
    idx = np.argsort(imp)
    ax.barh([feature_names[i] for i in idx], imp[idx], color=color, alpha=0.8)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Importancia', fontsize=9)
    ax.grid(True, axis='x', alpha=0.3)

plt.suptitle('Feature Importance Comparativa — DT · RF · XGBoost · Heart Disease',
             fontsize=13, fontweight='bold', color=UPCH_RED)
plt.tight_layout()
plt.savefig('fi_comparativa.png', dpi=150, bbox_inches='tight')
plt.show()
print('→ ¿Las tres metodologías coinciden en las features más importantes?')

---
# PARTE 4 — Comparación Final
## Ejercicio 6 — ROC curves y tabla de métricas

<div style='background:#fff3e0; padding:8px 14px; border-left:4px solid #e65100; border-radius:4px; margin:8px 0'>
📚 <b>TAREA</b> — completa esto antes de la siguiente clase
</div>

### Paso 6.1 — Curvas ROC

In [ ]:
# ── Curvas ROC: DT, RF, XGBoost ─────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 7))

for modelo, nombre, color in [
    (dt_opt,    f'Árbol de Decisión regularizado (depth={best_depth})', UPCH_BLUE),
    (rf_final,  'Random Forest (200 árboles)',              UPCH_RED),
    (xgb_final, f'XGBoost (lr={best_lr})',                 UPCH_GREEN)
]:
    # TODO 29: obtén las probabilidades de la clase positiva
    #   usa modelo.predict_proba(X_test)[:, 1]
    y_score = ___

    # TODO 30: calcula fpr, tpr con roc_curve y el AUC con auc()
    fpr, tpr, _ = ___
    roc_auc     = ___

    ax.plot(fpr, tpr, lw=2.5, color=color,
            label=f'{nombre}  (AUC = {roc_auc:.3f})')

ax.plot([0,1],[0,1], 'k--', lw=1.5, label='Clasificador aleatorio')
ax.set_xlabel('Tasa de Falsos Positivos  (1 − Especificidad)', fontsize=11)
ax.set_ylabel('Tasa de Verdaderos Positivos  (Sensibilidad)', fontsize=11)
ax.set_title('Curvas ROC — Heart Disease: DT · RF · XGBoost',
             fontsize=13, fontweight='bold', color=UPCH_RED)
ax.legend(fontsize=10, loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_comparativa.png', dpi=150, bbox_inches='tight')
plt.show()

### Paso 6.2 — Tabla de métricas comparativa

In [ ]:
# ── Tabla de métricas comparativa ───────────────────────────────
print('=' * 75)
print('     RESUMEN FINAL — Challenge 4: Árboles, RF y XGBoost')
print('=' * 75)
print(f'{"Modelo":<30} {"Accuracy":<12} {"Recall":<12} {"Precision":<12} {"F1"}')
print('-' * 75)

# TODO 31: completa la tabla con los valores de acc, rec, prec, f1
#           para DT, RF y XGBoost
resultados = [
    (f'Árbol de Decisión regularizado (d={best_depth})', ___, ___, ___, ___),
    ('Random Forest (n=200)',               ___, ___, ___, ___),
    (f'XGBoost (lr={best_lr})',             ___, ___, ___, ___),
]
for nombre, acc, rec, prec, f1 in resultados:
    print(f'{nombre:<30} {acc:.4f}       {rec:.4f}       {prec:.4f}       {f1:.4f}')

### Paso 6.3 — Interpretación sesgo-varianza

**📝 Tu respuesta (edita esta celda):**

Completa la tabla:

| Modelo | Sesgo | Varianza | Mecanismo de reducción |
|---|---|---|---|
| Árbol profundo | ___ | ___ | ___ |
| Árbol prepodado / profundidad controlada | ___ | ___ | ___ |
| Random Forest | ___ | ___ | ___ |
| XGBoost | ___ | ___ | ___ |

**¿En qué escenario clínico usarías cada modelo? Justifica:**

- **Árbol de Decisión prepodado:** ___
- **Random Forest:** ___
- **XGBoost:** ___



---
<div style='background:#f5f5f5; padding:15px; border-left:5px solid #8B0000; border-radius:4px'>
<b>Entrega:</b> Sube tu notebook ejecutado (.ipynb con outputs) a tu carpeta de GitHub del curso.<br>
<b>Nombre del archivo:</b> <code>Challenge4_ApellidoNombreDeAmbos.ipynb</code><br>
<b>Fecha límite:</b> antes de Clase 6
</div>
